# Análise de dados TCP-CII

In [81]:
import pandas as pd

In [82]:
df = pd.read_csv('./T CELL/DENV 1 - T Cell Prediction - Class II.csv')
df

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el core,netmhciipan_el score,netmhciipan_el percentile
0,1,QYKFQADSPKRLSAA,31,45,15,HLA-DRB3*01:01,7,0.08,FQADSPKRL,0.921387,0.08
1,1,KNETWKLARASFIEV,206,220,15,HLA-DRB1*09:01,42,0.30,WKLARASFI,0.801965,0.30
2,1,KNETWKLARASFIEV,206,220,15,HLA-DRB1*07:01,42,0.36,WKLARASFI,0.829795,0.36
3,1,QYKFQADSPKRLSAA,31,45,15,HLA-DRB3*02:02,7,0.56,FQADSPKRL,0.619942,0.56
4,1,KNETWKLARASFIEV,206,220,15,HLA-DRB1*01:01,42,0.57,WKLARASFI,0.871070,0.57
...,...,...,...,...,...,...,...,...,...,...,...
1831,1,WCCRSCTLPPLRFKG,311,325,15,HLA-DRB1*04:01,63,100.00,RSCTLPPLR,0.000074,100.00
1832,1,WCCRSCTLPPLRFKG,311,325,15,HLA-DRB1*04:05,63,100.00,CRSCTLPPL,0.000101,100.00
1833,1,WCCRSCTLPPLRFKG,311,325,15,HLA-DRB1*07:01,63,100.00,CRSCTLPPL,0.000269,100.00
1834,1,WCCRSCTLPPLRFKG,311,325,15,HLA-DRB1*09:01,63,100.00,CRSCTLPPL,0.000222,100.00


## Selecionando Epítopos com median binding percentile menor que 5.

In [83]:
df_mbp_m5 = df[df['median binding percentile'] < 5].copy()
df_mbp_m5

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el core,netmhciipan_el score,netmhciipan_el percentile
0,1,QYKFQADSPKRLSAA,31,45,15,HLA-DRB3*01:01,7,0.08,FQADSPKRL,0.921387,0.08
1,1,KNETWKLARASFIEV,206,220,15,HLA-DRB1*09:01,42,0.30,WKLARASFI,0.801965,0.30
2,1,KNETWKLARASFIEV,206,220,15,HLA-DRB1*07:01,42,0.36,WKLARASFI,0.829795,0.36
3,1,QYKFQADSPKRLSAA,31,45,15,HLA-DRB3*02:02,7,0.56,FQADSPKRL,0.619942,0.56
4,1,KNETWKLARASFIEV,206,220,15,HLA-DRB1*01:01,42,0.57,WKLARASFI,0.871070,0.57
...,...,...,...,...,...,...,...,...,...,...,...
62,1,TRLENIMWKQISNEL,61,75,15,HLA-DPA1*01:03/DPB1*02:01,13,4.60,LENIMWKQI,0.164233,4.60
63,1,ENDMKFTVVVGDVSG,81,95,15,HLA-DQA1*01:02/DQB1*06:02,17,4.80,MKFTVVVGD,0.277408,4.80
64,1,TTFIIDGPNTPECPD,131,145,15,HLA-DRB1*04:01,27,4.80,FIIDGPNTP,0.373010,4.80
65,1,VTNEVHTWTEQYKFQ,21,35,15,HLA-DQA1*04:01/DQB1*04:02,5,4.80,VHTWTEQYK,0.140050,4.80


## Agrupando por pepitideos e agregando colunas pertinentes.

In [84]:
epitopos_repetidos = (
    df_mbp_m5
    .groupby('peptide', as_index=False)
    .agg(
        start=("start", "first"),
        end=("end", "first"),
        qte_de_alelos=("allele", "nunique"),
        median_binding_percentile=(
            "median binding percentile",
            "median"
        ),
        alelos=(
            "allele",
            lambda x: ", ".join(sorted(x.unique()))
        )
    ))

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,ADMGYWIESEKNETW,196,210,1,2.900,HLA-DQA1*05:01/DQB1*02:01
1,ADSPKRLSAAIGKAW,36,50,1,3.700,HLA-DPA1*02:01/DPB1*14:01
2,ADVQNTTFIIDGPNT,126,140,4,2.750,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
3,AKIIGADVQNTTFII,121,135,3,3.000,"HLA-DQA1*01:02/DQB1*06:02, HLA-DQA1*05:01/DQB1..."
4,DSGCVINWKGRELKC,1,15,1,1.400,HLA-DRB1*15:01
5,EDGCWYGMEIRPVKE,326,340,1,3.300,HLA-DQA1*04:01/DQB1*04:02
6,EGTTVVVDEHCGNRG,281,295,1,2.200,HLA-DRB1*03:01
7,ENDMKFTVVVGDVSG,81,95,4,3.350,"HLA-DQA1*01:02/DQB1*06:02, HLA-DQA1*03:01/DQB1..."
8,FTVVVGDVSGILAQG,86,100,2,2.100,"HLA-DRB1*03:01, HLA-DRB3*01:01"
9,GIFTTNIWLKLRDSY,161,175,4,1.450,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."


# Filtragem por epítopos que presentes em mais de 2 alelos.

In [85]:
epitopos_repetidos = epitopos_repetidos[
    epitopos_repetidos["qte_de_alelos"] >= 2
].reset_index(drop=True)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,ADVQNTTFIIDGPNT,126,140,4,2.750,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
1,AKIIGADVQNTTFII,121,135,3,3.000,"HLA-DQA1*01:02/DQB1*06:02, HLA-DQA1*05:01/DQB1..."
2,ENDMKFTVVVGDVSG,81,95,4,3.350,"HLA-DQA1*01:02/DQB1*06:02, HLA-DQA1*03:01/DQB1..."
3,FTVVVGDVSGILAQG,86,100,2,2.100,"HLA-DRB1*03:01, HLA-DRB3*01:01"
4,GIFTTNIWLKLRDSY,161,175,4,1.450,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
5,GSGIFVTNEVHTWTE,16,30,3,2.800,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01, HLA..."
6,HKYSWKSWGKAKIIG,111,125,2,1.550,"HLA-DPA1*01:03/DPB1*02:01, HLA-DRB1*13:02"
7,ISNELNHILLENDMK,71,85,2,3.350,"HLA-DPA1*02:01/DPB1*01:01, HLA-DPA1*03:01/DPB1..."
8,ISQHNYRPGYFTQTA,251,265,2,3.650,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
9,KNETWKLARASFIEV,206,220,7,3.300,"HLA-DPA1*01:03/DPB1*04:01, HLA-DQA1*05:01/DQB1..."


In [86]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["median_binding_percentile", "qte_de_alelos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,GIFTTNIWLKLRDSY,161,175,4,1.450,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
1,HKYSWKSWGKAKIIG,111,125,2,1.550,"HLA-DPA1*01:03/DPB1*02:01, HLA-DRB1*13:02"
2,FTVVVGDVSGILAQG,86,100,2,2.100,"HLA-DRB1*03:01, HLA-DRB3*01:01"
3,YRPGYFTQTAGPWHL,256,270,2,2.120,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
4,QYKFQADSPKRLSAA,31,45,12,2.200,"HLA-DPA1*01:03/DPB1*04:01, HLA-DPA1*02:01/DPB1..."
5,ADVQNTTFIIDGPNT,126,140,4,2.750,"HLA-DPA1*01:03/DPB1*02:01, HLA-DPA1*01:03/DPB1..."
6,SKAVHADMGYWIESE,191,205,2,2.785,"HLA-DRB1*03:01, HLA-DRB3*01:01"
7,GSGIFVTNEVHTWTE,16,30,3,2.800,"HLA-DQA1*05:01/DQB1*02:01, HLA-DRB3*01:01, HLA..."
8,AKIIGADVQNTTFII,121,135,3,3.000,"HLA-DQA1*01:02/DQB1*06:02, HLA-DQA1*05:01/DQB1..."
9,TTFIIDGPNTPECPD,131,145,2,3.150,"HLA-DRB1*04:01, HLA-DRB3*01:01"
